# OPTIONAL - Act 6: Real-time inference on the Online Feature Store

> **Optional branch. Costs money (Postgres online store bills 24/7).** Only run this if you're
> showing real-time serving. Enable it the day before with `pre_demo/enable_online.py --yes`,
> and tear it down afterwards with `reset/teardown.py`.
>
> **Prerequisites (set in this notebook's Service settings):**
> - Online feature store is `RUNNING` (from `pre_demo/enable_online.py`).
> - Container runtime.
> - An **External Access Integration** + a **PAT secret** attached (online reads / REST need a PAT).
>   Set `PAT_SECRET` below to your secret's normalized path (e.g. `ml_fraud_dev_sandbox/feature_store/demo_pat`).

This shows the Postgres-backed online store: **millisecond point lookups**, **sub-2s stream
freshness**, and **real-time scoring** off online-served features.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ML_DEV_ROLE").collect()
session.sql("USE WAREHOUSE CORTEX_CODE_WH").collect()

DEV_DB, FS_SCHEMA, REG_SCHEMA = "ML_FRAUD_DEV_SANDBOX", "FEATURE_STORE", "ML"
KEY, MODEL = "ACCOUNT_ID", "AML_FRAUD_GBM"
FV_PROFILE, FV_VELOCITY, FV_VER, STREAM_SOURCE = "ACCOUNT_PROFILE", "ACCOUNT_VELOCITY", "V1", "TRANSACTION_EVENTS"
PAT_SECRET = "ml_fraud_dev_sandbox/feature_store/demo_pat"   # <-- your attached secret's path
VELOCITY_FEATURES = ["TXN_COUNT_1H","TXN_COUNT_24H","AMT_SUM_24H","AMT_SUM_48H",
                     "DISTINCT_BANKS_24H","DISTINCT_RECEIVERS_24H","CROSS_CCY_CNT_24H","HIGH_RISK_CNT_24H"]
session.sql(f"USE SCHEMA {DEV_DB}.{FS_SCHEMA}").collect()

import requests
from snowflake.snowpark.secrets import get_generic_secret_string
from snowflake.ml.feature_store import FeatureStore, CreationMode, StoreType, online_service

PAT = get_generic_secret_string(PAT_SECRET)
HEADERS = {"Authorization": f'Snowflake Token="{PAT}"', "Content-Type": "application/json"}
fs = FeatureStore(session=session, database=DEV_DB, name=FS_SCHEMA,
                  default_warehouse="CORTEX_CODE_WH", creation_mode=CreationMode.FAIL_IF_NOT_EXIST)
st = fs.get_online_service_status()
INGEST_URL = online_service.endpoint_url(st, "ingest")
QUERY_URL  = online_service.endpoint_url(st, "query")
print("online service:", getattr(st, "status", None))
print("ingest:", INGEST_URL, "| query:", QUERY_URL)

## 1. Millisecond point lookup (Postgres online store)

In [ ]:
import time
ACCOUNT = "012719_8019E5AE0"
fv = fs.get_feature_view(FV_PROFILE, FV_VER)
t0 = time.time()
df = fs.read_feature_view(fv, keys=[[ACCOUNT]], store_type=StoreType.ONLINE)
print(f"online point lookup: {(time.time()-t0)*1000:.0f} ms")
df

## 2. Stream ingest -> velocity features fresh in < 2s

In [ ]:
import random
from datetime import datetime, timedelta

def make_event(acct, seq):
    ts = (datetime.utcnow()+timedelta(milliseconds=seq)).strftime("%Y-%m-%d %H:%M:%S.%f")
    return {KEY: acct, "EVENT_TS": ts, "AMOUNT_PAID": round(random.uniform(50_000,900_000),2),
            "RECEIVER_ACCOUNT_ID": f"{random.randint(1,30000):05d}_{random.randint(0,9_000_000_000):010X}",
            "RECEIVER_BANK": f"{random.randint(1,250000)}", "IS_CROSS_CURRENCY": 1, "IS_HIGH_RISK_FORMAT": 1}

def query_velocity(acct):
    r = requests.post(f"{QUERY_URL}/api/v1/query", headers=HEADERS, json={
        "name": FV_VELOCITY, "version": FV_VER, "object_type": "feature_view",
        "request_rows": [{"entity": {KEY: acct}}]}, timeout=30)
    r.raise_for_status()
    return dict(zip(VELOCITY_FEATURES, r.json()["results"][0]["features"]))

MULE = f"NEWMULE_{int(time.time())}"
recs = [make_event(MULE, i) for i in range(40)]
for b in range(0, len(recs), 10):   # Ingest API caps records/request -> batch by 10
    requests.post(f"{INGEST_URL}/api/v1/ingest", headers=HEADERS,
                  json={"dry_run": False, "records": {STREAM_SOURCE: recs[b:b+10]}}, timeout=30)
time.sleep(3)
import pandas as pd
pd.DataFrame([query_velocity(MULE)], index=[MULE])   # counters jumped within ~2s

## 3. Real-time score off online-served features

In [ ]:
# Read the model + assemble its feature vector from the ONLINE profile lookup.
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name=DEV_DB, schema_name=REG_SCHEMA)
mv = reg.get_model(MODEL).default
fn = next(f for f in mv.show_functions() if f["target_method"].lower()=="predict_proba")
cols = [str(s.name).upper() for s in fn["signature"].inputs]

prof = fs.read_feature_view(fs.get_feature_view(FV_PROFILE, FV_VER),
                            keys=[[ACCOUNT]], store_type=StoreType.ONLINE)
prof.columns = [c.upper() for c in prof.columns]
row = {c: 0.0 for c in cols}
for c in cols:
    if c in prof.columns:
        row[c] = float(prof[c].iloc[0] or 0)
# suspicious request context + engineered risk from profile
row["AMOUNT_PAID"] = 9000; row["IS_CROSS_CURRENCY"] = 1; row["IS_CROSS_BORDER"] = 1
row["IS_HIGH_RISK_FORMAT"] = 0
if "PAYMENT_FORMAT_ACH" in row: row["PAYMENT_FORMAT_ACH"] = 1
if "HIST_RECEIVER_FANOUT" in row and "HIST_DISTINCT_RECEIVERS" in prof.columns:
    row["HIST_RECEIVER_FANOUT"] = float(prof.get("HIST_DISTINCT_RECEIVERS", pd.Series([0])).iloc[0] or 0)

import pandas as pd
X = pd.DataFrame([[row[c] for c in cols]], columns=cols)
scored = mv.run(X, function_name="predict_proba")
print("real-time risk score:", scored)

## What this shows
The **Postgres-backed Online Feature Store**: millisecond point lookups, continuous stream
aggregation with **sub-2s freshness**, and real-time scoring off online-served features — the
same feature definitions used offline for training (train/serve consistency).

**Tear down after the demo** to stop the 24/7 cost: `reset/teardown.py --yes`.